In [24]:
import time
import pandas as pd
import requests


# 1. Configuration
API_KEY = "m4o2IJsSYjMPtoyMxxCgSevpztowGUdZX7ByF0nypyk4Up5SW8WLYhbCKtLvF22e"  # Replace with your free API key
SEASON = 2024
OUTPUT_FILE = f"cbb_games_full_{SEASON}.csv"

headers = {"Authorization": f"Bearer {API_KEY}", "Accept": "application/json"}

# 2. Define monthly date ranges covering the entire CBB season (Nov - Apr)
date_ranges = [
    (f"{SEASON-1}-11-01", f"{SEASON-1}-11-15"),  # November
    (f"{SEASON-1}-11-16", f"{SEASON-1}-11-30"),
    (f"{SEASON-1}-12-01", f"{SEASON-1}-12-31"),  # December
    (f"{SEASON}-01-01", f"{SEASON}-01-31"),  # January
    (f"{SEASON}-02-01", f"{SEASON}-02-28"),  # February
    (f"{SEASON}-03-01", f"{SEASON}-04-30"),  # March & April (Postseason)
]

all_games = []
url = "https://api.collegebasketballdata.com/games/teams"

# 3. Fetch each chunk
for start_date, end_date in date_ranges:
    print(f"Fetching games from {start_date} to {end_date}...")

    params = {
        "season": SEASON,
        "startDateRange": start_date,
        "endDateRange": end_date,
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()
        print(f"  -> Retrieved {len(data)} games.")
        all_games.extend(data)
    else:
        print(
            f"  -> Error fetching range {start_date} to {end_date}: {response.status_code}"
        )

    # Pause briefly to respect API rate limits
    time.sleep(0.5)

# 4. Convert to DataFrame and deduplicate
df = pd.DataFrame(all_games)

# Deduplicate based on unique game id
if "id" in df.columns:
    df.drop_duplicates(subset=["id"], inplace=True)
else:
    if "id" in df.columns:
        df.drop_duplicates(subset=["id"], inplace=True)
    elif "game_id" in df.columns:
        df.drop_duplicates(subset=["game_id"], inplace=True)
    else:
        pass
        # If no ID exists, drop duplicate rows based on date + teams
        #df.drop_duplicates(subset=["date", "home_team", "away_team"], inplace=True)

# Save to CSV
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nFinished! Saved a total of {len(df)} games to '{OUTPUT_FILE}'.")

Fetching games from 2023-11-01 to 2023-11-15...
  -> Retrieved 1080 games.
Fetching games from 2023-11-16 to 2023-11-30...
  -> Retrieved 1526 games.
Fetching games from 2023-12-01 to 2023-12-31...
  -> Retrieved 2412 games.
Fetching games from 2024-01-01 to 2024-01-31...
  -> Retrieved 2700 games.
Fetching games from 2024-02-01 to 2024-02-28...
  -> Retrieved 2630 games.
Fetching games from 2024-03-01 to 2024-04-30...
  -> Retrieved 1708 games.

Finished! Saved a total of 12056 games to 'cbb_games_full_2024.csv'.


In [25]:
import pandas as pd
df = pd.read_csv("cbb_games_full_2024.csv")
#df.head()
print(df.columns.tolist())
df = df[["teamId", "team", "conference", "opponentId", "opponent", "opponentConference", "neutralSite", "isHome", "pace", "teamStats", "opponentStats"]].copy()
df.head()

['gameId', 'season', 'seasonLabel', 'seasonType', 'tournament', 'startDate', 'startTimeTbd', 'teamId', 'team', 'conference', 'teamSeed', 'opponentId', 'opponent', 'opponentConference', 'opponentSeed', 'neutralSite', 'isHome', 'conferenceGame', 'gameType', 'notes', 'gameMinutes', 'pace', 'teamStats', 'opponentStats']


,teamId,team,conference,opponentId,opponent,opponentConference,neutralSite,isHome,pace,teamStats,opponentStats
0,831,Spalding,NaN,115,IU Indianapolis,Horizon,False,False,69.0,"{'possessions': 69, 'assists': 10, 'steals': 6...","{'possessions': 69, 'assists': 9, 'steals': 10..."
1,115,IU Indianapolis,Horizon,831,Spalding,NaN,False,True,69.0,"{'possessions': 69, 'assists': 9, 'steals': 10...","{'possessions': 69, 'assists': 10, 'steals': 6..."
2,110,Hofstra,CAA,677,Saint Joseph's Long Island,NaN,False,True,71.0,"{'possessions': 71, 'assists': 28, 'steals': 8...","{'possessions': 71, 'assists': 11, 'steals': 5..."
3,677,Saint Joseph's Long Island,NaN,110,Hofstra,CAA,False,False,71.0,"{'possessions': 71, 'assists': 11, 'steals': 5...","{'possessions': 71, 'assists': 28, 'steals': 8..."
4,797,Fort Lauderdale,NaN,302,Troy,Sun Belt,False,False,72.5,"{'possessions': 72, 'assists': 4, 'steals': 8,...","{'possessions': 73, 'assists': 18, 'steals': 1..."


In [26]:
import ast
import pandas as pd


# 1. Safely convert string representations of dicts into actual Python dicts
def parse_dict(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return {}
    elif isinstance(val, dict):
        return val
    return {}


# Apply parser to columns
df["teamStats_clean"] = df["teamStats"].apply(parse_dict)
df["oppStats_clean"] = df["opponentStats"].apply(parse_dict)

# 2. Expand dictionaries AND add prefixes to prevent column name collisions
stats_df = pd.json_normalize(df["teamStats_clean"]).add_prefix("team_")
opp_df = pd.json_normalize(df["oppStats_clean"]).add_prefix("opp_")

# 3. Combine back with the original DataFrame (No collision error now!)
df = df.join(stats_df)
df = df.join(opp_df)

# 4. Clean up temporary / raw columns
df.drop(
    columns=["teamStats", "teamStats_clean", "oppStats_clean", "opponentStats"],
    inplace=True,
)

df.head()

,teamId,team,conference,opponentId,opponent,opponentConference,neutralSite,isHome,pace,team_possessions,...,opp_rebounds.offensive,opp_rebounds.defensive,opp_rebounds.total,opp_fouls.total,opp_fouls.technical,opp_fouls.flagrant,opp_fourFactors.effectiveFieldGoalPct,opp_fourFactors.freeThrowRate,opp_fourFactors.turnoverRatio,opp_fourFactors.offensiveReboundPct
0,831,Spalding,NaN,115,IU Indianapolis,Horizon,False,False,69.0,69.0,...,11.0,17.0,28.0,13.0,0.0,0.0,45.7,37.9,18.8,39.3
1,115,IU Indianapolis,Horizon,831,Spalding,NaN,False,True,69.0,69.0,...,12.0,20.0,32.0,17.0,0.0,0.0,44.1,22.0,29.0,37.5
2,110,Hofstra,CAA,677,Saint Joseph's Long Island,NaN,False,True,71.0,71.0,...,7.0,21.0,28.0,10.0,0.0,0.0,35.6,15.3,25.4,25.0
3,677,Saint Joseph's Long Island,NaN,110,Hofstra,CAA,False,False,71.0,71.0,...,16.0,31.0,47.0,12.0,0.0,0.0,64.9,3.9,11.3,34.0
4,797,Fort Lauderdale,NaN,302,Troy,Sun Belt,False,False,72.5,72.0,...,16.0,29.0,45.0,23.0,0.0,0.0,52.1,32.9,17.8,35.6


In [27]:
print(df.columns.tolist())
df = df[["teamId", "team", "conference", "opponentId", "opponent", "opponentConference", "neutralSite", "isHome", "pace", "team_points.total", "opp_points.total"]].copy()
df.head()

['teamId', 'team', 'conference', 'opponentId', 'opponent', 'opponentConference', 'neutralSite', 'isHome', 'pace', 'team_possessions', 'team_assists', 'team_steals', 'team_blocks', 'team_trueShooting', 'team_rating', 'team_gameScore', 'team_points.total', 'team_points.byPeriod', 'team_points.largestLead', 'team_points.fastBreak', 'team_points.inPaint', 'team_points.offTurnovers', 'team_twoPointFieldGoals.made', 'team_twoPointFieldGoals.attempted', 'team_twoPointFieldGoals.pct', 'team_threePointFieldGoals.made', 'team_threePointFieldGoals.attempted', 'team_threePointFieldGoals.pct', 'team_freeThrows.made', 'team_freeThrows.attempted', 'team_freeThrows.pct', 'team_fieldGoals.made', 'team_fieldGoals.attempted', 'team_fieldGoals.pct', 'team_turnovers.total', 'team_turnovers.teamTotal', 'team_rebounds.offensive', 'team_rebounds.defensive', 'team_rebounds.total', 'team_fouls.total', 'team_fouls.technical', 'team_fouls.flagrant', 'team_fourFactors.effectiveFieldGoalPct', 'team_fourFactors.free

,teamId,team,conference,opponentId,opponent,opponentConference,neutralSite,isHome,pace,team_points.total,opp_points.total
0,831,Spalding,NaN,115,IU Indianapolis,Horizon,False,False,69.0,63,70
1,115,IU Indianapolis,Horizon,831,Spalding,NaN,False,True,69.0,70,63
2,110,Hofstra,CAA,677,Saint Joseph's Long Island,NaN,False,True,71.0,101,48
3,677,Saint Joseph's Long Island,NaN,110,Hofstra,CAA,False,False,71.0,48,101
4,797,Fort Lauderdale,NaN,302,Troy,Sun Belt,False,False,72.5,47,92


In [28]:
import os
import sqlite3
import numpy as np
import pandas as pd

BASE_DIR = os.getcwd()
DB_PATH = os.path.join(BASE_DIR, "ratings.db")


def save_to_sqlite(output_df, season, db_path=DB_PATH):
  os.makedirs(os.path.dirname(db_path), exist_ok=True)

  # Create a clean copy to modify without affecting the original DataFrame
  df_to_save = output_df.copy()

  # Ensure team isn't duplicated if it's already a column
  if "team" not in df_to_save.columns:
    df_to_save = df_to_save.reset_index().rename(columns={"index": "team"})

  df_to_save["season"] = int(season)

  conn = sqlite3.connect(db_path)

  # Use 'replace' temporarily to rebuild the schema with the 'team' column,
  # or use 'append' if you are sure the table schema already matches.
  df_to_save.to_sql(
      "efficiency_ratings", conn, if_exists="append", index=False
  )
  conn.close()

  print(
      f"Successfully saved season {season} with columns {list(df_to_save.columns)} to: {db_path}"
  )

def calculate_iterative_sos(games_df, max_iterations=100, tolerance=0.0001):
    """Calculates iterative Strength of Schedule (SoS) adjusted Offensive Efficiency,

    Defensive Efficiency, Net Efficiency, and Pace using native CollegeBasketballData columns.
    """
    # -------------------------------------------------------------
    # 1. FILTER D1 VS D1 MATCHUPS & CLEAN TYPES
    # -------------------------------------------------------------
    # Create clean conference check columns
    games_df["Conference_clean"] = games_df["conference"].replace(
        r"^\s*$", np.nan, regex=True
    )
    games_df["OpponentConference_clean"] = games_df[
        "opponentConference"
    ].replace(r"^\s*$", np.nan, regex=True)

    # Keep only games where both teams belong to a recognized D1 conference
    games_df = games_df.dropna(
        subset=["Conference_clean", "OpponentConference_clean"]
    ).copy()

    # Enforce numeric data types to prevent calculation issues
    games_df["team_points.total"] = pd.to_numeric(
        games_df["team_points.total"], errors="coerce"
    )
    games_df["opp_points.total"] = pd.to_numeric(
        games_df["opp_points.total"], errors="coerce"
    )
    games_df["pace"] = pd.to_numeric(games_df["pace"], errors="coerce")

    # Drop any rows with bad or missing numeric values
    games_df = games_df.dropna(
        subset=["team_points.total", "opp_points.total", "pace"]
    ).copy()

    # Gather strictly D1 primary teams
    teams = games_df["team"].unique()

    # -------------------------------------------------------------
    # 2. CALCULATE LEAGUE AVERAGES
    # -------------------------------------------------------------
    total_points = float(
        (games_df["team_points.total"] + games_df["opp_points.total"]).sum()
    )
    total_possessions = float(
        (games_df["pace"] * 2).sum()
    )  # Total game possessions across both sides
    total_games = len(games_df)

    # League Average Efficiency per 100 possessions
    league_avg_eff = (total_points / total_possessions) * 100

    # League Average Pace per 40 minutes (average game possessions per team)
    league_avg_pace = float(games_df["pace"].mean())

    #print(f"League Average Efficiency: {league_avg_eff:.2f}")
    #print(f"League Average Pace (per 40m): {league_avg_pace:.2f}\n")

    # -------------------------------------------------------------
    # 3. INITIALIZE RAW TEAM METRICS
    # -------------------------------------------------------------
    team_stats = {}
    for team in teams:
        team_games = games_df[games_df["team"] == team]

        pts_scored = float(team_games["team_points.total"].sum())
        pts_allowed = float(team_games["opp_points.total"].sum())
        poss_total = float(team_games["pace"].sum())
        game_count = len(team_games)

        raw_oe = (pts_scored / poss_total) * 100
        raw_de = (pts_allowed / poss_total) * 100
        raw_pace = poss_total / game_count

        # Extract conference name for this team
        conf_name = team_games["Conference_clean"].iloc[0]

        # Keep only D1 opponents in the schedule tracker
        valid_opponents = [
            opp for opp in team_games["opponent"].tolist() if opp in teams
        ]

        team_stats[team] = {
            "team": team,
            "conference": conf_name,  # Added conference field
            "raw_oe": raw_oe,
            "raw_de": raw_de,
            "raw_pace": raw_pace,
            "adj_oe": raw_oe,
            "adj_de": raw_de,
            "adj_pace": raw_pace,
            "opponents": valid_opponents,
        }

    df_ratings = pd.DataFrame.from_dict(team_stats, orient="index")

    # -------------------------------------------------------------
    # 4. ITERATIVE SOS LOOP (OFFENSE, DEFENSE, & PACE)
    # -------------------------------------------------------------
    for iteration in range(1, max_iterations + 1):
        prev_adj_oe = df_ratings["adj_oe"].copy()

        new_adj_oe = {}
        new_adj_de = {}
        new_adj_pace = {}

        for team in teams:
            opponents = df_ratings.loc[team, "opponents"]

            if len(opponents) == 0:
                continue

            # Average ratings of opponents from previous pass
            opp_avg_de = df_ratings.loc[opponents, "adj_de"].mean()
            opp_avg_oe = df_ratings.loc[opponents, "adj_oe"].mean()
            opp_avg_pace = df_ratings.loc[opponents, "adj_pace"].mean()

            # Multipliers
            off_multiplier = league_avg_eff / opp_avg_de
            def_multiplier = league_avg_eff / opp_avg_oe
            pace_multiplier = league_avg_pace / opp_avg_pace

            # Update values
            new_adj_oe[team] = df_ratings.loc[team, "raw_oe"] * off_multiplier
            new_adj_de[team] = df_ratings.loc[team, "raw_de"] * def_multiplier
            new_adj_pace[team] = (
                df_ratings.loc[team, "raw_pace"] * pace_multiplier
            )

        df_ratings["adj_oe"] = pd.Series(new_adj_oe)
        df_ratings["adj_de"] = pd.Series(new_adj_de)
        df_ratings["adj_pace"] = pd.Series(new_adj_pace)

        # Check convergence
        max_change = np.max(np.abs(df_ratings["adj_oe"] - prev_adj_oe))

        if max_change < tolerance:
            print(
                f"Convergence reached after {iteration} iterations (Max change: {max_change:.6f})\n"
            )
            break

    # -------------------------------------------------------------
    # 5. SEPARATE OFFENSIVE & DEFENSIVE SOS
    # -------------------------------------------------------------
    df_ratings["net_eff"] = df_ratings["adj_oe"] - df_ratings["adj_de"]

    # Offensive SoS: Average opponent Adjusted DEFENSE faced
    df_ratings["off_sos"] = df_ratings["opponents"].apply(
        lambda opps: df_ratings.loc[opps, "adj_oe"].mean()
        if len(opps) > 0
        else np.nan
    )

    # Defensive SoS: Average opponent Adjusted OFFENSE faced
    df_ratings["def_sos"] = df_ratings["opponents"].apply(
        lambda opps: df_ratings.loc[opps, "adj_de"].mean()
        if len(opps) > 0
        else np.nan
    )

    # Overall SoS (Net): Average opponent Net Efficiency faced
    df_ratings["sos_net"] = df_ratings["opponents"].apply(
        lambda opps: df_ratings.loc[opps, "net_eff"].mean()
        if len(opps) > 0
        else np.nan
    )

    # Strip any non-D1 artifacts that contain NaN ratings
    df_ratings = df_ratings.dropna(subset=["adj_oe", "adj_de"]).copy()

    output_df = df_ratings[
        [
            "team",
            "conference",
            "net_eff",
            "adj_oe",
            "adj_de",
            "adj_pace",
            "sos_net",
            "off_sos",
            "def_sos",
        ]
    ].copy()
    output_df = output_df.sort_values(by="net_eff", ascending=False)
    output_df = output_df.round(2)
    save_to_sqlite(output_df, 2024)  
    return output_df


# Run script
calculate_iterative_sos(df)
#print(results.round(2).to_string())

Successfully saved season 2024 with columns ['team', 'conference', 'net_eff', 'adj_oe', 'adj_de', 'adj_pace', 'sos_net', 'off_sos', 'def_sos', 'season'] to: c:\Users\Josh Agrest\Desktop\agrest_analytics\app\ratings.db


,team,conference,net_eff,adj_oe,adj_de,adj_pace,sos_net,off_sos,def_sos
UConn,UConn,Big East,40.98,131.13,90.15,64.98,13.48,114.78,101.30
Houston,Houston,Big 12,40.10,124.14,84.05,64.01,15.70,115.19,99.48
Purdue,Purdue,Big Ten,38.39,130.87,92.48,67.11,18.05,117.01,98.95
Arizona,Arizona,Pac-12,36.65,127.37,90.72,72.86,14.84,115.45,100.61
Iowa State,Iowa State,Big 12,34.16,120.37,86.21,67.34,14.25,114.75,100.50
...,...,...,...,...,...,...,...,...,...
Stonehill,Stonehill,NEC,-28.66,89.92,118.58,68.12,-8.08,101.74,109.82
Lindenwood,Lindenwood,OVC,-28.73,90.82,119.55,67.66,-8.27,101.51,109.78
Coppin State,Coppin State,MEAC,-30.39,83.42,113.82,67.00,-8.58,102.28,110.86
IU Indianapolis,IU Indianapolis,Horizon,-31.03,90.26,121.29,67.60,-5.16,105.97,111.13
